## Распознавание именованных сущностей с использованием BERT

Основано на блокноте https://github.com/huggingface/notebooks/blob/master/examples/token_classification.ipynb

BERT (англ. Bidirectional Encoder Representations from Transformers – двунаправленный кодировщик представлений трансформера) — языковая модель, основанная на архитектуре Трансформер (Transformer), предназначенная для предобучения *языковых представлений* (Representation) с целью их последующего применения в широком спектре задач обработки естественного языка (NLP).

## Как работает BERT
BERT использует Трансформер – архитектуру, которая изучает контекстуальные отношения между словами в тексте. Она включает в себя два отдельных механизма — кодировщик, считывающий введенный текст, и декодер, выдающий прогноз. Поскольку целью BERT является создание языковой модели, он использует только механизм кодировщика.

В отличие от однонаправленных моделей, которые считывают вводимый текст последовательно (слева направо или справа налево), трансформер считывает сразу всю последовательность слов. Поэтому он считается двунаправленным, хотя правильнее было бы сказать, ненаправленным. Это свойство позволяет модели изучать контекст слова на основе всего его окружения.

Принцип работы трансформера представлен на рисунке. Вход представляет собой последовательность токенов (Token) (слов, их частей или символов), которые сначала преобразуются в векторы, а затем обрабатываются нейронной сетью. Выход представляет собой последовательность векторов, в которой каждый вектор соответствует входной лексеме с тем же индексом.
![alt text](https://resize.yandex.net/mailservice?url=https%3A%2F%2Fwww.helenkapatsa.ru%2Fcontent%2Fimages%2F2022%2F08%2Fimage.png&proxy=yes&key=91768e122e301805e3be976cd66cbc7d)

В этом блокноте модель дообучается на задаче классификации отдельных слов, а именно решается задача *распознавания именованных сущностей* (named entity recognition, *NER*). Используется датасет медицинских сущностей, но в целом пайплайн подходит для любой задачи на выделение сущностей в тексте.

In [ ]:
# Устанавливаем необходимые зависимости
# seqeval - фреймворк для оценки маркировки последовательности,
# используется для оценки эффективности в задачах разбиения на фрагменты, таких как распознавание именованных сущностей, разметка частей речи и др.

! pip install datasets transformers seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 486.2/486.2 kB 10.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 90.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 5.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 17.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.5/212.5 kB 29.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.3/134.3 kB 18.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 34.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 117.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 86.3 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16165 sha256=2bd1224a19e7fa30e6dbc5c074c4bef08f9a027d28bcedcc2ad5fb6156eab93a
  Stored in directory: /root/.cache/pip/wheels/1a/67/4a/ad4082dd7dfc30f2abfe4d80a2ed

Для скорости используем маленький BERT для русского языка [rubert-tiny](https://huggingface.co/cointegrated/rubert-tiny). Если взять другую, более крупную BERT-подобную модель, качество NER может быть выше, но и время обучения и работы будет дольше.


Этот ноутбук может быть использован для любой задачи классификации токенов с любой моделью из [Model Hub](https://huggingface.co/models), если у этой модели есть версия для классификации токенов с быстрым токенизатором (это можно проверить в [таблице](https://huggingface.co/transformers/index.html#bigtable)). Возможно, потребуются небольшие корректировки при использовании других наборов данных. В зависимости от модели и используемого графического процессора может потребоваться настроить размер пакета, чтобы избежать ошибок нехватки памяти.

In [ ]:
model_checkpoint = "cointegrated/rubert-tiny"
batch_size = 16

## Загрузка данных

Для обучения возьмём размеченный корпус русскоязычных отзывов на лекарства [Russian Drug Reaction Corpus](https://github.com/cimm-kzn/RuDReC).

Загрузим его библиотекой corus, потому что это удобно

In [ ]:
from datasets import load_dataset, load_metric

In [ ]:
!wget https://github.com/cimm-kzn/RuDReC/raw/master/data/rudrec_annotated.json
!pip install corus razdel

--2023-07-13 13:14:50--  https://github.com/cimm-kzn/RuDReC/raw/master/data/rudrec_annotated.json
Resolving github.com (github.com)... 192.30.255.113
Connecting to github.com (github.com)|192.30.255.113|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/cimm-kzn/RuDReC/master/data/rudrec_annotated.json [following]
--2023-07-13 13:14:51--  https://raw.githubusercontent.com/cimm-kzn/RuDReC/master/data/rudrec_annotated.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1773014 (1.7M) [text/plain]
Saving to: ‘rudrec_annotated.json’

rudrec_annotated.js 100%[===================>]   1.69M  --.-KB/s    in 0.06s   

2023-07-13 13:14:51 (27.6 MB/s) - ‘rudrec_annotated.json’ saved [1773014/1773014]

     

In [ ]:
from corus import load_rudrec
drugs = list(load_rudrec('rudrec_annotated.json'))
print(len(drugs))

4809


Пример документа:

In [ ]:
drugs[0]

RuDReCRecord(
    file_name='172744.tsv',
    text='нам прописали, так мой ребенок сыпью покрылся, глаза опухли, сверху и снизу на веках высыпала сыпь, ( 8 месяцев сыну)А от виферона такого не было... У кого ещё такие побочки, отзовитесь!1 Чем спасались?\n',
    sentence_id=0,
    entities=[RuDReCEntity(
         entity_id='*[0]_se',
         entity_text='виферона',
         entity_type='Drugform',
         start=122,
         end=130,
         concept_id='C0021735',
         concept_name=nan
     ), RuDReCEntity(
         entity_id='*[1]',
         entity_text='сыпью покрылся',
         entity_type='ADR',
         start=31,
         end=45,
         concept_id='C0015230',
         concept_name=nan
     ), RuDReCEntity(
         entity_id='*[2]',
         entity_text='глаза опухли',
         entity_type='ADR',
         start=47,
         end=59,
         concept_id='C4760994',
         concept_name=nan
     ), RuDReCEntity(
         entity_id='*[3]',
         entity_text='на веках высы

Посмотрим, какие сущности есть: лекарства, форма лекарств, класс лекарств, показания к применению, побочные действия и прочие болезни/симптомы.

https://arxiv.org/abs/2004.03659

* **DRUGNAME** - упоминание торговой марки препарата или ингредиентов/активных соединений продукта.
* **DRUGCLASS** - упоминание класса препарата, например, противовоспалительные или сердечно-сосудистые.
* **DRUGFORM** - упоминание о способах введения, например, таблетка или жидкость, которые описывают физическую форму, в которой лекарство будет доставлено в организм пациента.
* **DI** - любое указание/симптом, указывающий на причину
прием/назначение препарата.
* **ADR** - упоминание неблагоприятных побочных действий, которые происходят в результате приема лекарств и не связаны с симптомами лечения.
* **FINDING** - любой DI или ADR, которые не были непосредственно испытаны пациентом или членами его/ее семьи, или связаны с историей болезни/этикеткой препарата или любыми формами болезни, если аннотатор не ясно указал тип

In [ ]:
# Сollections - встроенный модуль Python для работы с данными контейнеронного типа
# Счетчик Counter позволяет вычислить частоту вхождений каждого элемента
# Возвращет словарь с элементами в качестве ключей и "счетчиком" (количеством вхождений элемента) в качестве значений
# most_common(n) возвращает в порядке убывания список n наиболее часто встречающихся элементов с указанием их количества
# defaultdict при обращении к отсутствующему ключу возвращает значение по умолчанию
# в остальном он ничем не отличается от обычного словаря

from collections import Counter, defaultdict
type2text = defaultdict(Counter)
ents = Counter()
for item in drugs:
    for e in item.entities:
        ents[e.entity_type] += 1
        type2text[e.entity_type][e.entity_text] += 1

for k, v in ents.most_common():
    print(k, v)
    print(type2text[k].most_common(3))

DI 1401
[('простуды', 64), ('ОРВИ', 47), ('профилактики', 42)]
Drugname 1043
[('Виферон', 33), ('Анаферон', 25), ('Циклоферон', 24)]
Drugform 836
[('таблетки', 154), ('таблеток', 79), ('свечи', 63)]
ADR 720
[('аллергия', 16), ('слабость', 13), ('диарея', 12)]
Drugclass 330
[('противовирусный', 21), ('противовирусное', 18), ('противовирусных', 13)]
Finding 236
[('аллергии', 12), ('температуры', 6), ('сонливости', 5)]


In [ ]:
drugs[0].text

'нам прописали, так мой ребенок сыпью покрылся, глаза опухли, сверху и снизу на веках высыпала сыпь, ( 8 месяцев сыну)А от виферона такого не было... У кого ещё такие побочки, отзовитесь!1 Чем спасались?\n'

Напишем функцию, перекладывающую разметку сущностей на уровень слов. Будем использовать [IOB](https://en.wikipedia.org/wiki/Inside–outside–beginning_(tagging))-нотацию, чтобы разделять несколько сущностей одного типа, идущих подряд.

In [ ]:
from razdel import tokenize

def extract_labels(item):
    raw_toks = list(tokenize(item.text))
    words = [tok.text for tok in raw_toks]
    word_labels = ['O'] * len(raw_toks)
    char2word = [None] * len(item.text)
    for i, word in enumerate(raw_toks):
        char2word[word.start:word.stop] = [i] * len(word.text)

    for e in item.entities:
        e_words = sorted({idx for idx in char2word[e.start:e.end] if idx is not None})
        word_labels[e_words[0]] = 'B-' + e.entity_type
        for idx in e_words[1:]:
            word_labels[idx] = 'I-' + e.entity_type

    return {'tokens': words, 'tags': word_labels}

In [ ]:
print(extract_labels(drugs[0]))

{'tokens': ['нам', 'прописали', ',', 'так', 'мой', 'ребенок', 'сыпью', 'покрылся', ',', 'глаза', 'опухли', ',', 'сверху', 'и', 'снизу', 'на', 'веках', 'высыпала', 'сыпь', ',', '(', '8', 'месяцев', 'сыну', ')', 'А', 'от', 'виферона', 'такого', 'не', 'было', '...', 'У', 'кого', 'ещё', 'такие', 'побочки', ',', 'отзовитесь', '!', '1', 'Чем', 'спасались', '?'], 'tags': ['O', 'O', 'O', 'O', 'O', 'O', 'B-ADR', 'I-ADR', 'O', 'B-ADR', 'I-ADR', 'O', 'O', 'O', 'O', 'B-ADR', 'I-ADR', 'I-ADR', 'I-ADR', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-Drugform', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']}


In [ ]:
from sklearn.model_selection import train_test_split
ner_data = [extract_labels(item) for item in drugs]
ner_train, ner_test = train_test_split(ner_data, test_size=0.2, random_state=1)

Пример данных

In [ ]:
import pandas as pd
pd.options.display.max_colwidth = 300
pd.DataFrame(ner_train).sample(3)

,tokens,tags
3352,"[Время, использования, :, 2013, год]","[O, O, O, O, O]"
2684,"[бесполезно, !!!]","[O, O]"
1238,"[Еще, он, мне, хорошо, помогала, при, спазмах, в, кишечнике, и, желудке, ,, вызывая, при, этом, здоровый, аппетит, .]","[O, O, O, O, O, O, B-DI, I-DI, I-DI, I-DI, I-DI, O, O, O, O, O, B-DI, O]"


Соберём все виды меток в список.

In [ ]:
label_list = sorted({label for item in ner_train for label in item['tags']})
if 'O' in label_list:
    label_list.remove('O')
    label_list = ['O'] + label_list
label_list

['O',
 'B-ADR',
 'B-DI',
 'B-Drugclass',
 'B-Drugform',
 'B-Drugname',
 'B-Finding',
 'I-ADR',
 'I-DI',
 'I-Drugclass',
 'I-Drugform',
 'I-Drugname',
 'I-Finding']

Сложим наши данные в объект [`DatasetDict`](https://huggingface.co/docs/datasets/package_reference/main_classes.html#datasetdict), нативный для huggingface.

In [ ]:
from datasets import Dataset, DatasetDict

In [ ]:
ner_data = DatasetDict({
    'train': Dataset.from_pandas(pd.DataFrame(ner_train)),
    'test': Dataset.from_pandas(pd.DataFrame(ner_test))
})
ner_data

DatasetDict({
    train: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 3847
    })
    test: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 962
    })
})

## Preprocessing the data

Прежде чем мы сможем найти эти тексты для нашей модели, нам
нужно их предварительно обработать. Это делается с помощью токенизатора Transformers, который (как следует из названия) маркирует входные данные
(включая преобразование токенов в их соответствующие идентификаторы в
предварительно подготовленном словаре) и помещает их в формат,
ожидаемый моделью, а также генерирует другие входные данные, которые
требуются модели.

Чтобы это сделать, мы создадим экземпляр нашего токенизатора с
помощью метода AutoTokenizer.from_pretrained, который обеспечит:

- мы получаем токенизатор, соответствующий архитектуре модели,
которую мы хотим использовать,
- мы загружаем словарь, используемый при предварительном обучении
этой конкретной контрольной точке.

Этот словарь будет вызван, поэтому он не будет загружен снова при
следующем запуске ячейки.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

Downloading:   0%|          | 0.00/341 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/632 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/241k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/468k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/112 [00:00<?, ?B/s]

You can directly call this tokenizer on one sentence:

In [ ]:
tokenizer("Hello, this is one sentence!")

{'input_ids': [2, 9944, 16, 881, 550, 835, 15503, 5, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}

Depending on the model you selected, you will see different keys in the dictionary returned by the cell above. They don't matter much for what we're doing here (just know they are required by the model we will instantiate later), you can learn more about them in [this tutorial](https://huggingface.co/transformers/preprocessing.html) if you're interested.

Так как в нашем случае входные данные уже были разделены на слова,
мы должны передать список слов в свой токенайзер с аргументом `is_split_into_words=True`:

In [ ]:
tokenizer(["Hello", ",", "this", "is", "one", "sentence", "split", "into", "words", "."], is_split_into_words=True)

{'input_ids': [2, 9944, 16, 881, 550, 835, 15503, 7440, 996, 6301, 18, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

Обратим внимание, что трансформаторы часто предварительно
обучаются с помощью токенизаторов вложенных слов, что означает, что даже если входные данные уже были разделены на слова, каждое из этих
слов может быть снова разделено токенизатором.

In [ ]:
example = ner_train[5]
print(example["tokens"])

['Мы', 'поменяли', 'место', 'жительства', 'и', 'перевели', 'дочь', 'в', 'школу', ',', 'которая', 'находится', 'ближе', 'к', 'дому', '.']


In [ ]:
tokenized_input = tokenizer(example["tokens"], is_split_into_words=True)
tokens = tokenizer.convert_ids_to_tokens(tokenized_input["input_ids"])
print(tokens)

['[CLS]', 'Мы', 'пом', '##ен', '##яли', 'место', 'ж', '##итель', '##ства', 'и', 'пер', '##еве', '##ли', 'дочь', 'в', 'школу', ',', 'которая', 'находится', 'б', '##ли', '##же', 'к', 'дому', '.', '[SEP]']


Чтобы перейти с уровня слов на уровень subword tokens, нужно ещё раз предобработать тексты.

In [ ]:
len(example["tags"]), len(tokenized_input["input_ids"])

(16, 26)

К счастью, токенизатор возвращает выходные данные, которые имеют
метод `word_ids`, который может нам помочь.

In [ ]:
print(tokenized_input.word_ids())

[None, 0, 1, 1, 1, 2, 3, 3, 3, 4, 5, 5, 5, 6, 7, 8, 9, 10, 11, 12, 12, 12, 13, 14, 15, None]


Как мы можем заметить, токенизатор возвращает список с тем же
количеством элементов, что и наши обработанные входные идентификаторы,
сопоставляя специальные токены с None и все остальные токены с их
соответствующим словом. Таким образом, мы можем выровнять метки с
обработанными входными идентификаторами.

In [ ]:
word_ids = tokenized_input.word_ids()
aligned_labels = [-100 if i is None else example["tags"][i] for i in word_ids]
print(len(aligned_labels), len(tokenized_input["input_ids"]))

NameError: ignored

Здесь мы устанавливаем метки всех специальных токенов равными -
100 (индекс, который игнорируется PyTorch), а метки всех остальных токенов – метке слова, из которого они происходят. Другая стратегия заключается в том, чтобы установить метку только для первого токена, полученного из данного слова, и присвоить метку -100 другим подтокенам из того же слова. Мы предлагаем здесь две стратегии, просто измените флаг `label_all_tokens`.

Теперь мы готовы написать функцию, которая будет предварительно
обрабатывать наши образцы. Мы отправляем их в токенизатор с аргументом
truncation=True (для усечения текстов, размер которых превышает
максимальный размер, разрешенный моделью) и is_split_into_words=True
(как показано выше). Затем мы выравниваем метки с идентификаторами
токенов, используя выбранную нами стратегию:

In [ ]:
def tokenize_and_align_labels(examples, label_all_tokens=True):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(examples['tags']):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            # Special tokens have a word id that is None. We set the label to -100 so they are automatically
            # ignored in the loss function.
            if word_idx is None:
                label_ids.append(-100)
            # We set the label for the first token of each word.
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            # For the other tokens in a word, we set the label to either the current label or -100, depending on
            # the label_all_tokens flag.
            else:
                label_ids.append(label[word_idx] if label_all_tokens else -100)
            previous_word_idx = word_idx

        label_ids = [label_list.index(idx) if isinstance(idx, str) else idx for idx in label_ids]

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

Эта функция работает с одним или несколькими примерами. В случае
нескольких примеров токенизатор вернет список для каждого ключа:

In [ ]:
tokenize_and_align_labels(ner_data['train'][22:23])

{'input_ids': [[2, 1041, 4033, 3236, 9267, 331, 19173, 19106, 26629, 1887, 22018, 548, 22276, 320, 21538, 16, 705, 13718, 22264, 548, 18397, 14063, 11137, 626, 16296, 24531, 18, 3]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], 'labels': [[-100, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 7, 7, 7, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -100]]}

Чтобы применить эту функцию ко всем предложениям (или парам
предложений) в нашем наборе данных, мы просто используем метод map
нашего объекта dataset, который мы создали ранее. Это применит функцию
ко всем элементам всех разбиений в dataset, так что наши обучающие,
валидационные и тестовые данные будут предварительно обработаны одной
командой.

In [ ]:
tokenized_datasets = ner_data.map(tokenize_and_align_labels, batched=True)

NameError: ignored

Результаты автоматически кэшируются библиотекой наборов данных, что позволяет не тратить время на этот шаг при следующем запуске. Библиотека Datasets обычно способна определить, когда функция, передаваемая в `map`, изменилась (и, следовательно, требует не использовать данные кэша). Datasets предупреждает вас, когда использует кэшированные файлы, можно передать `load_from_cache_file=False` в вызове `map`, чтобы не использовать кэшированные файлы и принудительно применить предварительную обработку снова.

Обратите внимание, что передали `batched=True` для совместного кодирования текстов пакетами. Это сделано для того, чтобы в полной мере использовать преимущества быстрого токенизатора, который был загружен ранее, который будет использовать многопоточность для одновременной обработки текстов в пакете.

## Fine-tuning the model

После подготовки данных можно загрузить предварительно подготовленную модель и точно настроить ее. Поскольку задачи связаны с классификацией токенов, используем класс `AutoModelForTokenClassification`. Как и в случае с токенизатором, метод `from_pretrained` загрузит и кэширует модель. Единственное, что требуется указать, это количество меток для задачи (которые можно получить из функций, как показано ранее):

In [ ]:
label_list

['O',
 'B-ADR',
 'B-DI',
 'B-Drugclass',
 'B-Drugform',
 'B-Drugname',
 'B-Finding',
 'I-ADR',
 'I-DI',
 'I-Drugclass',
 'I-Drugform',
 'I-Drugname',
 'I-Finding']

In [ ]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer

model = AutoModelForTokenClassification.from_pretrained(model_checkpoint, num_labels=len(label_list))
model.config.id2label = dict(enumerate(label_list))
model.config.label2id = {v: k for k, v in model.config.id2label.items()}

Downloading:   0%|          | 0.00/47.7M [00:00<?, ?B/s]

Some weights of the model checkpoint at cointegrated/rubert-tiny were not used when initializing BertForTokenClassification: ['cls.predictions.decoder.bias', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForTokenClassification were not initialized f

Предупреждение говорит о том, что мы отбрасываем некоторые веса
(слои `vocab_transform` и vocab_layer_norm) и случайным образом
инициализируем некоторые другие (слои `pre_classifier` и `classifier`). В данном
случае это абсолютно нормально, потому что мы удаляем head, используемый для предварительной обработки модели в задаче моделирования на замаскированном языке, и заменяем его новым head, для которого у нас нет предварительно обработанных весов, поэтому библиотека предупреждает нас, что мы должны точно настроить эту модель, прежде чем использовать ее для вывода.

Чтобы создать экземпляр `Trainer`, нужно определить еще три
вещи. Наиболее важным является [`TrainingArguments`](https://huggingface.co/transformers/main_classes/trainer.html#transformers.TrainingArguments), который представляет собой класс, содержащий все атрибуты для настройки обучения. Для этого требуется имя папки, которое будет использоваться для сохранения
контрольных точек модели, а все остальные аргументы необязательны:

In [ ]:
args = TrainingArguments(
    "ner",
    evaluation_strategy = "epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=10,
    weight_decay=0.01,
    save_strategy='no',
    report_to='none',
)

Далее устанавливаем оценку, которая будет выполняться в конце
каждой эпохи, настраиваем скорость обучения, используем `batch_size`,
определенный в верхней части блокнота, и настраиваем количество эпох для
обучения, а также уменьшение веса.

Также понадобится средство сбора данных, которое будет группировать обработанные примеры вместе, применяя отступы, чтобы сделать их все одинакового размера (каждый блокнот будет дополнен до длины самого длинного примера). Для этой задачи в библиотеке Transformers есть сборщик данных, который передает не только входные данные, но и метки.

In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

И последнее, что нужно определить для тренировочной выборки
– это как вычислять показатели на основе прогнозов. Для этого загрузим
метрику [`seqeval`](https://github.com/chakki-works/seqeval) (которая обычно используется для оценки результатов в наборе данных CONLL) через библиотеку Datasets.

In [ ]:
metric = load_metric("seqeval")

Downloading:   0%|          | 0.00/2.48k [00:00<?, ?B/s]

Эта метрика принимает список меток для прогнозов и ссылок:

In [ ]:
example = ner_train[4]
labels = example['tags']
metric.compute(predictions=[labels], references=[labels])

{'DI': {'f1': 1.0, 'number': 1, 'precision': 1.0, 'recall': 1.0},
 'Drugform': {'f1': 1.0, 'number': 2, 'precision': 1.0, 'recall': 1.0},
 'overall_accuracy': 1.0,
 'overall_f1': 1.0,
 'overall_precision': 1.0,
 'overall_recall': 1.0}

Поэтому необходимо провести небольшую постобработку предсказаний:
- выбираем прогнозируемый индекс (с максимальным логитом) для
каждого токена;
- преобразуем его в строковую метку;
- игнорируем везде, где мы устанавливаем метку -100.

Следующая функция выполняет всю эту постобработку результата
`Trainer.evaluate` (который представляет собой `namedtuple`, содержащий
прогнозы и метки) перед применением метрики.

In [ ]:
import numpy as np

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (special tokens)
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels, zero_division=0)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

Обратим внимание, что мы отбрасываем precision/recall/f1,
вычисленные для каждой категории, и фокусируемся только на общих
precision/recall/f1.
Затем просто нужно передать все это вместе с наборами данных тренеру `Trainer`:

In [ ]:
trainer = Trainer(
    model,
    args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.evaluate()

The following columns in the evaluation set  don't have a corresponding argument in `BertForTokenClassification.forward` and have been ignored: tokens, tags.
***** Running Evaluation *****
  Num examples = 962
  Batch size = 16


{'eval_accuracy': 0.07571226846083562,
 'eval_f1': 0.03137110167927662,
 'eval_loss': 2.604278326034546,
 'eval_precision': 0.018480269594521145,
 'eval_recall': 0.10372178157413056,
 'eval_runtime': 1.5067,
 'eval_samples_per_second': 638.492,
 'eval_steps_per_second': 40.486}

В начале обучения заморозим все параметры в модели, кроме последнего слоя, и посмотрим, насколько хорошо она обучится.

In [ ]:
for param in model.bert.parameters():
    param.requires_grad = False

In [ ]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)
        print(param)

classifier.weight
Parameter containing:
tensor([[-5.3295e-02,  8.1591e-05, -1.4091e-02,  ...,  9.4435e-03,
          2.6371e-02, -2.7459e-02],
        [-1.4154e-02,  1.8980e-02, -6.4149e-03,  ..., -3.0063e-02,
         -8.0335e-03, -1.3474e-02],
        [ 3.9226e-03, -1.7339e-03, -2.4043e-03,  ...,  1.1911e-02,
         -6.8623e-03, -3.6764e-02],
        ...,
        [ 2.9699e-02, -2.5830e-02,  2.9956e-03,  ...,  2.0724e-02,
          2.6304e-02, -1.3127e-04],
        [-2.8258e-02,  1.9521e-03, -1.2629e-02,  ..., -2.4292e-02,
         -1.9133e-02,  3.5226e-02],
        [ 4.8563e-03, -3.9019e-02,  2.2573e-02,  ...,  2.3094e-02,
         -5.4334e-03, -3.1281e-02]], device='cuda:0', requires_grad=True)
classifier.bias
Parameter containing:
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], device='cuda:0',
       requires_grad=True)


Теперь можно точно настроить модель, просто вызвав метод `train`:

In [ ]:
import logging
from transformers.trainer import logger as noisy_logger
noisy_logger.setLevel(logging.WARNING)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,2.034866,0.032157,0.059487,0.041747,0.630991
2,No log,1.594469,0.042105,0.004881,0.008748,0.815724
3,2.043100,1.282439,0.052632,0.000305,0.000607,0.826314
4,2.043100,1.079169,0.000000,0.000000,0.000000,0.826854
5,1.267200,0.954540,0.000000,0.000000,0.000000,0.826896
6,1.267200,0.880644,0.000000,0.000000,0.000000,0.826896
7,0.932500,0.837882,0.000000,0.000000,0.000000,0.826896
8,0.932500,0.813664,0.000000,0.000000,0.000000,0.826896
9,0.808700,0.801121,0.000000,0.000000,0.000000,0.826896
10,0.808700,0.797258,0.000000,0.000000,0.000000,0.826896


TrainOutput(global_step=2410, training_loss=1.181212188594074, metrics={'train_runtime': 31.5523, 'train_samples_per_second': 1219.246, 'train_steps_per_second': 76.381, 'total_flos': 35752217175750.0, 'train_loss': 1.181212188594074, 'epoch': 10.0})

Модель недообучилась, вероятно, нужно обучить больше слоёв.

Чтобы получить precision/recall/f1, вычисленную для каждой категории, можно применить ту же функцию, что и раньше, к результату метода `predict`:

In [ ]:
predictions, labels, _ = trainer.predict(tokenized_datasets["test"])
predictions = np.argmax(predictions, axis=2)

# Remove ignored index (special tokens)
true_predictions = [
    [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]
true_labels = [
    [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]

results = metric.compute(predictions=true_predictions, references=true_labels)
results

/usr/local/lib/python3.7/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


{'ADR': {'f1': 0.30279898218829515,
  'number': 446,
  'precision': 0.35,
  'recall': 0.26681614349775784},
 'DI': {'f1': 0.493963782696177,
  'number': 821,
  'precision': 0.4207369323050557,
  'recall': 0.5980511571254568},
 'Drugclass': {'f1': 0.7868852459016393,
  'number': 336,
  'precision': 0.7880597014925373,
  'recall': 0.7857142857142857},
 'Drugform': {'f1': 0.7922794117647058,
  'number': 565,
  'precision': 0.8240917782026769,
  'recall': 0.7628318584070797},
 'Drugname': {'f1': 0.8734309623430963,
  'number': 918,
  'precision': 0.8400402414486922,
  'recall': 0.9095860566448801},
 'Finding': {'f1': 0.0, 'number': 192, 'precision': 0.0, 'recall': 0.0},
 'overall_accuracy': 0.9050170279923582,
 'overall_f1': 0.6448696700316409,
 'overall_precision': 0.6370943733253944,
 'overall_recall': 0.652837095790116}

In [ ]:
from sklearn.metrics import confusion_matrix
import pandas as pd

In [ ]:
cm = pd.DataFrame(
    confusion_matrix(sum(true_labels, []), sum(true_predictions, []), labels=label_list),
    index=label_list,
    columns=label_list
)
cm

,O,B-ADR,B-DI,B-Drugclass,B-Drugform,B-Drugname,B-Finding,I-ADR,I-DI,I-Drugclass,I-Drugform,I-Drugname,I-Finding
O,19494,29,175,35,60,71,0,20,26,0,0,0,0
B-ADR,159,135,133,8,2,0,0,4,5,0,0,0,0
B-DI,242,21,525,0,17,10,0,3,3,0,0,0,0
B-Drugclass,50,1,17,264,0,4,0,0,0,0,0,0,0
B-Drugform,98,4,11,1,432,17,0,1,1,0,0,0,0
B-Drugname,44,1,16,1,8,848,0,0,0,0,0,0,0
B-Finding,56,32,87,5,3,3,0,1,5,0,0,0,0
I-ADR,180,51,40,0,1,0,0,47,30,0,0,0,0
I-DI,236,17,102,10,0,1,0,11,46,0,0,0,0
I-Drugclass,0,0,0,4,0,0,0,0,0,0,0,0,0


In [ ]:
model.save_pretrained('ner_bert.bin')
tokenizer.save_pretrained('ner_bert.bin')

Configuration saved in ner_bert.bin/config.json
Model weights saved in ner_bert.bin/pytorch_model.bin
tokenizer config file saved in ner_bert.bin/tokenizer_config.json
Special tokens file saved in ner_bert.bin/special_tokens_map.json


('ner_bert.bin/tokenizer_config.json',
 'ner_bert.bin/special_tokens_map.json',
 'ner_bert.bin/vocab.txt',
 'ner_bert.bin/added_tokens.json',
 'ner_bert.bin/tokenizer.json')

# Применение модели

In [ ]:
import torch

In [ ]:
text = ' '.join(ner_train[8]['tokens'])
text = ' '.join(ner_test[4]['tokens'])
text

'Охотно применяю его при борьбе с насморком , что в моем случае явление очень частое .'

In [ ]:
import torch

In [ ]:
tokens = tokenizer(text, return_tensors='pt')
tokens = {k: v.to(model.device) for k, v in tokens.items()}

with torch.no_grad():
    pred = model(**tokens)
pred.logits.shape

torch.Size([1, 29, 13])

In [ ]:
indices = pred.logits.argmax(dim=-1)[0].cpu().numpy()
token_text = tokenizer.convert_ids_to_tokens(tokens['input_ids'][0])
for t, idx in zip(token_text, indices):
    print(f'{t:15s} {label_list[idx]:10s}')

[CLS]           O         
О               O         
##хо            O         
##тно           O         
при             O         
##мен           O         
##я             O         
##ю             O         
его             O         
при             O         
борьбе          O         
с               O         
нас             B-DI      
##мор           B-DI      
##ком           B-DI      
,               O         
что             O         
в               O         
м               O         
##ое            O         
##м             O         
случае          O         
я               O         
##вление        O         
очень           O         
часто           O         
##е             O         
.               O         
[SEP]           O         


Более простое применение модели: пайплайн от huggingface

In [ ]:
from transformers import pipeline

In [ ]:
pipe = pipeline(model=model, tokenizer=tokenizer, task='ner', aggregation_strategy='average', device=0)

In [ ]:
print(text)
print(pipe(text))

Охотно применяю его при борьбе с насморком , что в моем случае явление очень частое .
[{'entity_group': 'DI', 'score': 0.73669535, 'word': 'насморком', 'start': 33, 'end': 42}]


**Задание**. Модель недообучилась, вероятно, нужно обучить больше слоёв.
Разморозьте несколько верхних или все слои (param.requires_grad = True) и дообучите модель. При этом, возможно, потребуется увеличить количество эпох обучения. Можно также поэспериментировать с другими гиперпараметрами (batch_size, learning_rate и др.)